# POC — Cross-sectional return regression

This notebook explores whether the current canonical feature set is better suited to predicting
continuous forward returns than the existing binary classification targets.

It is intentionally a **proof of concept**, not a new modeling framework:

- reuse the repository's canonical dataset, target, and temporal-split contracts;
- fit notebook-local Ridge and XGBoost regressors;
- never access the locked test period;
- evaluate prediction quality primarily as a daily cross-sectional ranking problem;
- compare multiple forward-return horizons without changing the generated feature set.

The main questions are:

1. Can the models rank future returns at all?
2. Is any apparent signal pooled across dates, or present within each signal date?
3. How are realized returns distributed among the top-ranked stocks each day?
4. Is the ranking relationship monotonic across predicted-return quantiles?
5. Are observed top-k results distinguishable from date-matched random ranking?


## 1. Configuration

The default configuration discovers all tickers present in the local bronze price table for
`yfinance`. Narrow `TICKERS` manually when a faster smoke test is useful.

The outer test period remains declared because it is part of the repository's split contract, but
this notebook never reads or evaluates it.


In [ ]:
from datetime import date
from pathlib import Path

PROVIDER = "yfinance"

# Set to a tuple such as ("AAK.ST", "SAAB-B.ST", "VOLV-B.ST") for a quick smoke test.
# Leave as None to discover all locally stored tickers for PROVIDER.
TICKERS: tuple[str, ...] | None = None

DATA_CUTOFF = date(2026, 7, 24)

TRAIN_START = date(2000, 1, 1)
TRAIN_END = date(2022, 12, 31)
VALIDATION_START = date(2023, 1, 1)
VALIDATION_END = date(2024, 12, 31)
TEST_START = date(2025, 1, 1)
TEST_END = DATA_CUTOFF

HORIZONS = (5, 10, 15)
RANDOM_SEED = 42
BOOTSTRAP_ITERATIONS = 500
PERMUTATION_ITERATIONS = 500

# Keep the first run inexpensive. Increase after the notebook has completed successfully.
XGB_PARAMS = {
    "n_estimators": 120,
    "learning_rate": 0.03,
    "max_depth": 4,
    "min_child_weight": 5,
    "gamma": 1.0,
    "subsample": 0.8,
    "colsample_bytree": 0.8,
    "reg_lambda": 2.0,
    "reg_alpha": 0.5,
    "objective": "reg:squarederror",
    "eval_metric": "rmse",
    "tree_method": "hist",
    "n_jobs": -1,
    "random_state": RANDOM_SEED,
}


## 2. Build the canonical dataset and fixed outer split

In [ ]:
import pandas as pd
from sqlalchemy import text

from swingtrader.data.db import resolve_database_engine
from swingtrader.data.features import DEFAULT_FEATURE_SET
from swingtrader.modeling.datasets import (
    UniverseSpec,
    FORWARD_RETURN_PRIMARY_TASK,
    FORWARD_RETURN_TARGET_SET,
    build_temporal_dataset,
)
from swingtrader.modeling.experiments import (
    ExperimentSpec,
    FixedTemporalSplitter,
    ModelSpec,
    TemporalSplitSpec,
)
from swingtrader.modeling.training import LOGISTIC_REGRESSION_MODEL_TYPE

repo_root = next(path for path in [Path.cwd(), *Path.cwd().parents] if (path / "pyproject.toml").exists())
database_url = f"sqlite+pysqlite:///{(repo_root / 'data' / 'swingtrader.sqlite').as_posix()}"
engine = resolve_database_engine(database_url=database_url)

if TICKERS is None:
    with engine.connect() as connection:
        ticker_rows = connection.execute(
            text(
                '''
                SELECT DISTINCT ticker
                FROM bronze_market_daily_prices
                WHERE provider = :provider
                ORDER BY ticker
                '''
            ),
            {"provider": PROVIDER},
        ).fetchall()
    resolved_tickers = tuple(row[0] for row in ticker_rows)
else:
    resolved_tickers = tuple(TICKERS)

if not resolved_tickers:
    raise RuntimeError(
        f"No bronze tickers were found for provider {PROVIDER!r}. "
        "Populate the local database or set TICKERS explicitly."
    )

len(resolved_tickers), resolved_tickers[:10]


In [ ]:
feature_set = DEFAULT_FEATURE_SET

# ModelSpec is required by ExperimentSpec, but this notebook does not route regression through the
# production baseline harness. The declared model is therefore only a stable experiment identity.
placeholder_model = ModelSpec(
    name="notebook_regression_poc",
    version="1",
    model_type=LOGISTIC_REGRESSION_MODEL_TYPE,
    hyperparameters={},
    feature_columns=None,
)

universe = UniverseSpec(
    name="local_bronze_regression_universe",
    version="1",
    provider=PROVIDER,
    tickers=resolved_tickers,
)

split_spec = TemporalSplitSpec(
    name="regression_poc_fixed_holdout",
    version="1",
    train_start=TRAIN_START,
    train_end=TRAIN_END,
    validation_start=VALIDATION_START,
    validation_end=VALIDATION_END,
    test_start=TEST_START,
    test_end=TEST_END,
)

experiment_spec = ExperimentSpec(
    name="return_regression_poc",
    version="1",
    feature_set=feature_set,
    target_set=FORWARD_RETURN_TARGET_SET,
    task=FORWARD_RETURN_PRIMARY_TASK,
    universe=universe,
    data_start=TRAIN_START,
    data_end=TEST_END,
    split=split_spec,
    model=placeholder_model,
    random_seeds={"model": RANDOM_SEED, "evaluation": RANDOM_SEED + 1},
)

bundle = build_temporal_dataset(engine=engine, spec=experiment_spec.dataset_spec)
split_result = FixedTemporalSplitter(experiment_spec.split).assign(bundle)

{
    "experiment_digest": experiment_spec.digest,
    "ticker_count": len(resolved_tickers),
    "generated_feature_count": len(bundle.manifest.feature_columns),
    "target_columns": bundle.manifest.target_columns,
    "outer_train": split_result.summary("train").to_manifest(),
    "outer_validation": split_result.summary("validation").to_manifest(),
}


## 3. Extract train and validation frames

`FORWARD_RETURN_TARGET_SET` already generates `forward_return_5d`, `forward_return_10d`, and
`forward_return_15d`. The selected V1 task only determines canonical dataset inclusion, so each
horizon is filtered independently below.

The locked-test indices are deliberately never requested.


In [ ]:
import numpy as np

feature_columns = tuple(bundle.manifest.feature_columns)
train_positions = split_result.indices("train")
validation_positions = split_result.indices("validation")

X_all = bundle.features.loc[:, feature_columns]
targets_all = bundle.targets
metadata_all = bundle.samples

def make_horizon_frames(horizon: int):
    target_column = f"forward_return_{horizon}d"
    if target_column not in targets_all.columns:
        raise KeyError(
            f"{target_column!r} is unavailable. Available targets: "
            f"{tuple(targets_all.columns)}"
        )

    def build(positions):
        X = X_all.iloc[positions].copy()
        y = pd.to_numeric(
            targets_all.iloc[positions][target_column],
            errors="coerce",
        )
        metadata = metadata_all.iloc[positions].copy()

        valid = y.notna() & np.isfinite(y)
        X = X.loc[valid]
        y = y.loc[valid].astype("float64")
        metadata = metadata.loc[valid]

        frame = metadata.copy()
        frame["actual_return"] = y
        return X, y, frame

    return (*build(train_positions), *build(validation_positions))

horizon_shapes = {}
for horizon in HORIZONS:
    X_train, y_train, train_frame, X_validation, y_validation, validation_frame = (
        make_horizon_frames(horizon)
    )
    horizon_shapes[horizon] = {
        "train_rows": len(y_train),
        "validation_rows": len(y_validation),
        "train_mean_return": y_train.mean(),
        "validation_mean_return": y_validation.mean(),
    }

pd.DataFrame(horizon_shapes).T


## 4. Notebook-local preprocessing and regressors

Both models use train-only median imputation. Ridge also uses train-only standardization.
XGBoost consumes the imputed values directly.

No early stopping or validation-based tree selection is used in this POC.


In [ ]:
from sklearn.impute import SimpleImputer
from sklearn.linear_model import Ridge
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from xgboost import XGBRegressor

def make_models():
    ridge = Pipeline(
        steps=[
            ("imputer", SimpleImputer(strategy="median", keep_empty_features=True)),
            ("scaler", StandardScaler()),
            ("regressor", Ridge(alpha=10.0)),
        ]
    )
    xgboost_median_imputed = Pipeline(
        steps=[
            ("imputer", SimpleImputer(strategy="median", keep_empty_features=True)),
            ("regressor", XGBRegressor(**XGB_PARAMS)),
        ]
    )
    xgboost_native = XGBRegressor(**XGB_PARAMS)
    return {
        "ridge": ridge,
        "xgboost_native": xgboost_native,
        "xgboost_median_imputed": xgboost_median_imputed,
    }


## 5. Evaluation helpers

The notebook reports conventional regression metrics, but the primary diagnostics are:

- pooled Pearson, Spearman, and Kendall correlation;
- mean and median daily cross-sectional Spearman/Kendall correlation;
- daily top-k realized-return distributions;
- predicted-return deciles and the top-minus-bottom spread;
- date-block bootstrap confidence intervals;
- within-date score permutations as a matched random-ranking baseline.


In [ ]:
from math import ceil

from scipy.stats import kendalltau, pearsonr, spearmanr
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

DATE_LEVEL = "trading_date"

def finite_correlation(function, x, y):
    mask = np.isfinite(x) & np.isfinite(y)
    if mask.sum() < 3:
        return np.nan
    result = function(np.asarray(x)[mask], np.asarray(y)[mask])
    return float(result.statistic if hasattr(result, "statistic") else result[0])

def add_predictions(frame, predictions):
    predictions = np.asarray(
        predictions,
        dtype="float64",
    ).reshape(-1)

    if len(predictions) != len(frame):
        raise ValueError(
            "Prediction count does not match frame rows: "
            f"{len(predictions)} != {len(frame)}"
        )
    result = frame.copy()
    result["predicted_return"] = predictions

    if (
        DATE_LEVEL not in result.index.names
        and DATE_LEVEL not in result.columns
    ):
        raise KeyError(
            f"{DATE_LEVEL!r} is unavailable in metadata "
            "or the index."
        )
    return result

def aggregate_metrics(frame):
    actual = frame["actual_return"].to_numpy()
    predicted = frame["predicted_return"].to_numpy()
    return {
        "rows": len(frame),
        "mae": mean_absolute_error(actual, predicted),
        "rmse": mean_squared_error(actual, predicted) ** 0.5,
        "r2": r2_score(actual, predicted),
        "pearson": finite_correlation(pearsonr, predicted, actual),
        "spearman": finite_correlation(spearmanr, predicted, actual),
        "kendall_tau": finite_correlation(kendalltau, predicted, actual),
    }

def daily_information_coefficients(frame):
    rows = []
    for signal_date, group in frame.groupby(DATE_LEVEL, sort=True):
        rows.append(
            {
                DATE_LEVEL: signal_date,
                "candidate_count": len(group),
                "spearman": finite_correlation(
                    spearmanr,
                    group["predicted_return"],
                    group["actual_return"],
                ),
                "kendall_tau": finite_correlation(
                    kendalltau,
                    group["predicted_return"],
                    group["actual_return"],
                ),
            }
        )
    return pd.DataFrame(rows)

def summarize_daily_information_coefficients(daily_ic):
    return {
        "dates": len(daily_ic),
        "mean_daily_spearman": daily_ic["spearman"].mean(),
        "median_daily_spearman": daily_ic["spearman"].median(),
        "daily_spearman_positive_fraction": daily_ic["spearman"].gt(0).mean(),
        "mean_daily_kendall": daily_ic["kendall_tau"].mean(),
        "median_daily_kendall": daily_ic["kendall_tau"].median(),
        "daily_kendall_positive_fraction": daily_ic["kendall_tau"].gt(0).mean(),
    }

def select_daily_top(frame, label):
    selected_groups = []
    for _, group in frame.groupby(DATE_LEVEL, sort=True):
        ordered = group.sort_values("predicted_return", ascending=False)
        if label.startswith("top_") and label.endswith("pct"):
            fraction = float(label.removeprefix("top_").removesuffix("pct")) / 100
            count = max(1, ceil(len(ordered) * fraction))
        elif label.startswith("top_"):
            count = int(label.removeprefix("top_"))
        else:
            raise ValueError(f"Unsupported selection label: {label}")
        selected_groups.append(ordered.head(min(count, len(ordered))))
    return pd.concat(selected_groups)

def return_distribution(frame, selection):
    selected = select_daily_top(frame, selection)
    returns = selected["actual_return"]
    full_mean = frame["actual_return"].mean()
    return {
        "selection": selection,
        "selected_rows": len(selected),
        "positive_rows": int(returns.gt(0).sum()),
        "mean_return": returns.mean(),
        "median_return": returns.median(),
        "std_return": returns.std(),
        "p05": returns.quantile(0.05),
        "p10": returns.quantile(0.10),
        "p25": returns.quantile(0.25),
        "p75": returns.quantile(0.75),
        "p90": returns.quantile(0.90),
        "p95": returns.quantile(0.95),
        "positive_fraction": returns.gt(0).mean(),
        "full_universe_mean": full_mean,
        "mean_return_spread": returns.mean() - full_mean,
    }

SELECTIONS = ("top_1", "top_3", "top_5", "top_1pct", "top_2pct", "top_5pct")


In [ ]:
def daily_quantile_table(frame, quantiles=10):
    rows = []
    for signal_date, group in frame.groupby(DATE_LEVEL, sort=True):
        if group["predicted_return"].nunique() < 2:
            continue
        ranked = group["predicted_return"].rank(method="first")
        bucket = pd.qcut(ranked, q=min(quantiles, len(group)), labels=False, duplicates="drop")
        daily = group.assign(prediction_bucket=bucket)
        for prediction_bucket, bucket_group in daily.groupby("prediction_bucket"):
            rows.append(
                {
                    DATE_LEVEL: signal_date,
                    "prediction_bucket": int(prediction_bucket) + 1,
                    "rows": len(bucket_group),
                    "mean_actual_return": bucket_group["actual_return"].mean(),
                    "median_actual_return": bucket_group["actual_return"].median(),
                    "positive_fraction": bucket_group["actual_return"].gt(0).mean(),
                    "mean_predicted_return": bucket_group["predicted_return"].mean(),
                }
            )
    result = pd.DataFrame(rows)
    return (
        result.groupby("prediction_bucket", as_index=False)
        .agg(
            dates=(DATE_LEVEL, "nunique"),
            rows=("rows", "sum"),
            mean_actual_return=("mean_actual_return", "mean"),
            median_actual_return=("median_actual_return", "median"),
            positive_fraction=("positive_fraction", "mean"),
            mean_predicted_return=("mean_predicted_return", "mean"),
        )
    )

def top_bottom_spread(quantile_table):
    bottom = quantile_table.iloc[0]["mean_actual_return"]
    top = quantile_table.iloc[-1]["mean_actual_return"]
    return float(top - bottom)


In [ ]:
def date_block_bootstrap(
    frame,
    selection,
    iterations=BOOTSTRAP_ITERATIONS,
    seed=RANDOM_SEED,
):
    rng = np.random.default_rng(seed)

    if DATE_LEVEL in frame.index.names:
        dates = pd.Index(frame.index.get_level_values(DATE_LEVEL).unique())
        grouped = {key: value for key, value in frame.groupby(level=DATE_LEVEL, sort=False)}
    elif DATE_LEVEL in frame.columns:
        dates = pd.Index(frame[DATE_LEVEL].drop_duplicates())
        grouped = {key: value for key, value in frame.groupby(DATE_LEVEL, sort=False)}
    else:
        raise KeyError(f"{DATE_LEVEL!r} is unavailable in the frame.")

    estimates = []

    for _ in range(iterations):
        sampled_dates = rng.choice(dates.to_numpy(), size=len(dates), replace=True)
        sampled_parts = []

        for sample_id, signal_date in enumerate(sampled_dates):
            part = grouped[signal_date].copy()

            # A date may appear multiple times in the bootstrap sample.
            # Give each occurrence a unique synthetic date so the
            # daily selector treats them as separate resampled days.
            part["_bootstrap_date"] = sample_id
            sampled_parts.append(part)

        sampled = pd.concat(sampled_parts, ignore_index=False)
        sampled = sampled.reset_index()
        sampled[DATE_LEVEL] = sampled["_bootstrap_date"]
        selected = select_daily_top(sampled, selection)
        estimates.append(selected["actual_return"].mean())

    estimates = np.asarray(estimates, dtype="float64")

    point_estimate = select_daily_top(frame, selection)["actual_return"].mean()

    return {
        "selection": selection,
        "mean_return": point_estimate,
        "bootstrap_ci_low": np.quantile(
            estimates,
            0.025,
        ),
        "bootstrap_ci_high": np.quantile(
            estimates,
            0.975,
        ),
    }

def within_date_permutation_test(
    frame,
    selection,
    iterations=PERMUTATION_ITERATIONS,
    seed=RANDOM_SEED,
):
    rng = np.random.default_rng(seed)
    observed = select_daily_top(frame, selection)["actual_return"].mean()
    null_estimates = []

    for _ in range(iterations):
        permuted = frame.copy()
        permuted["predicted_return"] = (
            permuted.groupby(DATE_LEVEL, sort=False)["predicted_return"]
            .transform(lambda values: rng.permutation(values.to_numpy()))
        )
        null_estimates.append(select_daily_top(permuted, selection)["actual_return"].mean())

    null_estimates = np.asarray(null_estimates)
    p_value = (1 + np.sum(null_estimates >= observed)) / (iterations + 1)
    return {
        "selection": selection,
        "observed_mean_return": observed,
        "null_mean_return": null_estimates.mean(),
        "null_p95": np.quantile(null_estimates, 0.95),
        "one_sided_p_value": p_value,
    }


## 6. Fit Ridge and XGBoost for each horizon

This can take several minutes on the full local universe because the canonical feature set is
generated once and each horizon fits two models.


In [ ]:
from tqdm import tqdm

In [ ]:
model_results = {}
prediction_frames = {}

for horizon in HORIZONS:
    (
        X_train,
        y_train,
        train_frame,
        X_validation,
        y_validation,
        validation_frame,
    ) = make_horizon_frames(horizon)

    for model_name, model in tqdm(make_models().items()):
        key = (horizon, model_name)
        model.fit(X_train, y_train)
        validation_predictions = model.predict(X_validation)
        scored = add_predictions(validation_frame, validation_predictions)

        aggregate = aggregate_metrics(scored)
        daily_ic = daily_information_coefficients(scored)
        aggregate.update(summarize_daily_information_coefficients(daily_ic))

        model_results[key] = {
            **aggregate,
            "horizon": horizon,
            "model": model_name,
        }
        prediction_frames[key] = scored

aggregate_results = (
    pd.DataFrame(model_results.values())
    .set_index(["horizon", "model"])
    .sort_index()
)
aggregate_results


## 7. Daily top-k realized-return distributions

A useful cross-sectional model should improve the mean and preferably the median realized return
of the highest-ranked stocks. Inspect tail quantiles as well: an attractive mean driven by a few
extreme winners is not a stable selection edge.


In [ ]:
top_k_rows = []
for (horizon, model_name), frame in prediction_frames.items():
    for selection in SELECTIONS:
        top_k_rows.append(
            {
                "horizon": horizon,
                "model": model_name,
                **return_distribution(frame, selection),
            }
        )

top_k_results = (
    pd.DataFrame(top_k_rows)
    .set_index(["horizon", "model", "selection"])
    .sort_index()
)
top_k_results


## 8. Predicted-return quantiles and monotonicity

In [ ]:
quantile_tables = {}
quantile_summary_rows = []

for key, frame in prediction_frames.items():
    horizon, model_name = key
    table = daily_quantile_table(frame, quantiles=10)
    quantile_tables[key] = table
    quantile_summary_rows.append(
        {
            "horizon": horizon,
            "model": model_name,
            "bottom_decile_mean_return": table.iloc[0]["mean_actual_return"],
            "top_decile_mean_return": table.iloc[-1]["mean_actual_return"],
            "top_minus_bottom_spread": top_bottom_spread(table),
            "bucket_return_spearman": finite_correlation(
                spearmanr,
                table["prediction_bucket"],
                table["mean_actual_return"],
            ),
        }
    )

quantile_summary = (
    pd.DataFrame(quantile_summary_rows)
    .set_index(["horizon", "model"])
    .sort_index()
)
quantile_summary


In [ ]:
# Change this key to inspect another horizon/model combination.
INSPECT_KEY = (5, "xgboost_native")
quantile_tables[INSPECT_KEY]


## 9. Uncertainty and matched random-ranking baseline

Run these tests first for the most promising horizon/model combination rather than multiplying
computation across every candidate immediately.

The bootstrap resamples complete signal dates. The permutation test keeps each day's candidates
and actual returns intact while randomly reassigning model scores within that date.


In [ ]:
uncertainty_rows = []
permutation_rows = []

frame_to_test = prediction_frames[INSPECT_KEY]
# for selection in SELECTIONS:
#     uncertainty_rows.append(date_block_bootstrap(frame_to_test, selection))
#     permutation_rows.append(within_date_permutation_test(frame_to_test, selection))

# bootstrap_results = pd.DataFrame(uncertainty_rows).set_index("selection")
# permutation_results = pd.DataFrame(permutation_rows).set_index("selection")

# bootstrap_results.join(permutation_results, rsuffix="_permutation")


In [ ]:
# bootstrap_results = pd.DataFrame(uncertainty_rows).set_index("selection")
# permutation_results = pd.DataFrame(permutation_rows).set_index("selection")

# bootstrap_results.join(permutation_results, rsuffix="_permutation")

In [ ]:
# bootstrap_results.join(permutation_results, rsuffix="_permutation").reset_index().to_dict("records")

In [ ]:
top_5_permutation = within_date_permutation_test(
    frame_to_test,
    "top_5",
    iterations=500,
    seed=RANDOM_SEED,
    # show_progress=True,
)

top_5_permutation

## 10. Visual diagnostics

The first plot compares predicted and realized returns on a random validation sample. The second
shows realized return by daily predicted-return decile. The third shows the time series of daily
Spearman information coefficients.

Do not interpret a visually narrow scatter as failure by itself; ranking metrics and top-k return
distributions are the primary diagnostics.


In [ ]:
import matplotlib.pyplot as plt

plot_frame = prediction_frames[(5, "xgboost_median_imputed")]
sample = plot_frame.sample(min(5_000, len(plot_frame)), random_state=RANDOM_SEED)

plt.figure(figsize=(9, 6))
plt.scatter(
    sample["predicted_return"],
    sample["actual_return"],
    alpha=0.2,
    s=10,
)
plt.axhline(0, linewidth=1)
plt.axvline(0, linewidth=1)
plt.xlabel("Predicted forward return")
plt.ylabel("Actual forward return")
plt.title(f"Predicted vs actual returns — {(5, "xgboost_median_imputed")}")
plt.show()


In [ ]:
deciles = quantile_tables[INSPECT_KEY]

plt.figure(figsize=(9, 5))
plt.plot(
    deciles["prediction_bucket"],
    deciles["mean_actual_return"],
    marker="o",
)
plt.axhline(0, linewidth=1)
plt.xlabel("Daily predicted-return decile (low to high)")
plt.ylabel("Mean actual forward return")
plt.title(f"Cross-sectional decile curve — {INSPECT_KEY}")
plt.show()


In [ ]:
daily_ic = daily_information_coefficients(prediction_frames[(5, "xgboost_median_imputed")])

plt.figure(figsize=(11, 5))
plt.plot(daily_ic[DATE_LEVEL], daily_ic["spearman"], linewidth=0.8)
plt.axhline(0, linewidth=1)
plt.xlabel("Signal date")
plt.ylabel("Daily Spearman correlation")
plt.title(f"Daily cross-sectional information coefficient — {(5, "xgboost_median_imputed")}")
plt.show()


## 11. Interpretation checklist

Use the outputs above to answer these questions before expanding the POC:

1. **Absolute regression quality**
   - Are MAE/RMSE better than a constant-mean prediction?
   - Is validation R² positive? A negative R² does not automatically reject a ranking model.

2. **Pooled versus cross-sectional signal**
   - Is pooled Spearman positive while mean daily Spearman is approximately zero?
   - If so, the model is mostly identifying favorable dates or regimes, not stocks within dates.

3. **Top-k usefulness**
   - Do daily top 1/3/5 or top 1%/2%/5% groups have higher mean and median realized returns?
   - Are downside quantiles acceptable?
   - Do bootstrap intervals exclude the full-universe mean?

4. **Matched null baseline**
   - Does the observed top-k mean exceed the within-date permutation distribution?
   - Treat isolated low p-values cautiously when many horizons, models, and cutoffs are inspected.

5. **Monotonicity**
   - Do realized returns generally increase from the lowest to highest prediction decile?
   - Is the top-minus-bottom spread positive and stable?

6. **Direction forward**
   - Regression succeeds cross-sectionally: compare its selected stocks directly with classifier rankings.
   - Regression succeeds only pooled: prioritize market/regime context and separate date-level from stock-level modeling.
   - Neither formulation succeeds: reconsider features, universe, target horizon, or target construction before framework work.

The locked test period must remain untouched until the target, model family, feature schema, and
selection rule are frozen.


In [ ]:
frame = prediction_frames[INSPECT_KEY]
print(INSPECT_KEY)

prediction_diagnostics = (
    frame.groupby(level=DATE_LEVEL)["predicted_return"]
    .agg(
        candidate_count="size",
        unique_predictions="nunique",
        minimum="min",
        maximum="max",
        standard_deviation="std",
    )
)

prediction_diagnostics.describe()

In [ ]:
def count_max_score_ties(group):
    maximum = group["predicted_return"].max()
    return pd.Series(
        {
            "candidate_count": len(group),
            "unique_predictions": group["predicted_return"].nunique(),
            "max_score_count": group["predicted_return"].eq(maximum).sum(),
            "max_score": maximum,
        }
    )


max_score_ties = (
    frame.groupby(level=DATE_LEVEL)
    .apply(count_max_score_ties, include_groups=False)
)

max_score_ties.describe()

In [ ]:
def top_score_bucket_size(group):
    maximum = group["predicted_return"].max()

    return pd.Series(
        {
            "candidate_count": len(group),
            "unique_predictions": group["predicted_return"].nunique(),
            "top_bucket_size": group["predicted_return"].eq(maximum).sum(),
            "top_bucket_fraction": group["predicted_return"].eq(maximum).mean(),
        }
    )


top_bucket_diagnostics = (
    frame.groupby(level=DATE_LEVEL)
    .apply(top_score_bucket_size, include_groups=False)
)

top_bucket_diagnostics.describe()

In [ ]:
top_bucket_diagnostics[
    "top_bucket_size"
].value_counts().sort_index()

In [ ]:
frame["predicted_return"].value_counts().head(20)

In [ ]:
def select_daily_top(frame, label, *, skip_unranked_dates=True):
    selected_groups = []

    for _, group in frame.groupby(level=DATE_LEVEL, sort=True):
        eligible = group.loc[
            np.isfinite(group["predicted_return"])
            & np.isfinite(group["actual_return"])
        ].copy()

        if eligible.empty:
            continue

        if (skip_unranked_dates and eligible["predicted_return"].nunique() == 1):
            continue

        eligible["_ticker_sort"] = eligible.index.get_level_values("ticker")

        ordered = eligible.sort_values(
            ["predicted_return", "_ticker_sort"],
            ascending=[False, True],
            kind="mergesort",
        )

        if label.startswith("top_") and label.endswith("pct"):
            fraction = float(label.removeprefix("top_").removesuffix("pct")) / 100
            count = max(1, ceil(len(ordered) * fraction))
        else:
            count = int(label.removeprefix("top_"))

        selected_groups.append(ordered.head(count).drop(columns="_ticker_sort"))

    if not selected_groups:
        return frame.iloc[0:0].copy()

    return pd.concat(selected_groups)

In [ ]:
top_1_all_dates = select_daily_top(frame, "top_1", skip_unranked_dates=False)
top_1_signal_dates = select_daily_top(frame, "top_1", skip_unranked_dates=True)

pd.DataFrame(
    {
        "all_dates": {
            "dates": top_1_all_dates.index
                .get_level_values(DATE_LEVEL)
                .nunique(),
            "mean_return": top_1_all_dates["actual_return"].mean(),
            "median_return": top_1_all_dates["actual_return"].median(),
        },
        "signal_dates_only": {
            "dates": top_1_signal_dates.index
                .get_level_values(DATE_LEVEL)
                .nunique(),
            "mean_return": top_1_signal_dates["actual_return"].mean(),
            "median_return": top_1_signal_dates["actual_return"].median(),
        },
    }
).T

In [ ]:
def classify_top_bucket(group):
    top_score = group["predicted_return"].max()
    top_bucket_size = group["predicted_return"].eq(top_score).sum()

    if group["predicted_return"].nunique() == 1:
        return "no_signal"

    if top_bucket_size == 1:
        return "unique_top"

    return "tied_top"


date_classification = (
    frame.groupby(level=DATE_LEVEL)
    .apply(classify_top_bucket)
    .rename("ranking_state")
)

In [ ]:
top_1 = select_daily_top(frame, "top_1", skip_unranked_dates=False).copy()
dates = top_1.index.get_level_values(DATE_LEVEL)

top_1["ranking_state"] = dates.map(date_classification)

top_1.groupby("ranking_state")["actual_return"].agg(
    ["count", "mean", "median", "std"]
)